# BUS-BRA: standalone Kaggle experiment

**Status: initial scaffold, not a completed runnable experiment.** Configuration
and schema cells are executable; the execution boundary deliberately raises
`NotImplementedError`. No data preparation, GPU pilot, or inference is claimed.

Owner: BUS-BRA lead. Run one model/configuration per execution.
This notebook will contain all required code and must not import project modules
or depend on another notebook's execution. Dataset/model access remains external.

## 1. Inputs and dependency setup

Attach BUS-BRA and, for inference, the approved frozen manifest to Kaggle. The
local archive `Datasets/BUS-BRA.zip` is excluded from Git; its entries include
`BUSBRA/Images/`, masks, and `BUSBRA/bus_data.csv`. Set `DATA_ROOT` to the extracted
`BUSBRA` directory when implemented. Kaggle attachment names/paths may differ.

Metadata header observed: `ID, Case, Histology, Pathology, BIRADS, Device, Width,
Height, Side, BBOX`. Verify the meanings and label mapping during the audit.
Masks, BI-RADS, histology, bounding boxes, and patient metadata are never model inputs.

**TODO:** add pinned dependency installation after GPU compatibility is tested.
No model packages are installed by this scaffold. All helpers will live in this notebook.

## 2. Configuration and shared output fields

Edit the settings here. Never select a final configuration using test accuracy.

In [ ]:
from pathlib import Path

DATASET = 'bus_bra'
MODEL_KEY = "qwen2_vl"      # qwen2_vl, medvlm_r1, gemma3, medgemma
RUN_MODE = "pilot"         # prepare, pilot, evaluate
RUN_ID = None              # Assign a unique ID before a real execution.
SEED = 42
BATCH_SIZE = 1
DO_SAMPLE = False
MAX_NEW_TOKENS = 32       # Initial agreed cap; revise only before final freeze.
TARGET_TEST_SIZE = 400  # Subject to audited eligibility.
PILOT_SIZE = 20         # Gemma pilot only; Qwen pilot count remains pending.

# Configure Kaggle attachment paths explicitly; never embed access tokens here.
DATA_ROOT = None           # Extracted image/metadata root.
MANIFEST_PATH = None       # A frozen pilot/test manifest, not freshly sampled per model.
OUTPUT_ROOT = Path("/kaggle/working/bus_bra")

MODEL_REVISION = None      # Exact model commit; floating main is not final-ready.
PROCESSOR_REVISION = None  # Exact processor commit.
PRECISION = None           # fp16, fp32, or int4_fp32 after selecting pilot configuration.
IMPLEMENTATION_VALIDATED = False  # Remains false until code and checks are complete.

MODEL_SPECS = {
    "qwen2_vl": {"model_id": "Qwen/Qwen2-VL-2B-Instruct", "family": "qwen2_vl"},
    "medvlm_r1": {"model_id": "JZPeterPan/MedVLM-R1", "family": "qwen2_vl"},
    "gemma3": {"model_id": "google/gemma-3-4b-it", "family": "gemma3"},
    "medgemma": {"model_id": "google/medgemma-4b-it", "family": "gemma3"},
}

MANIFEST_FIELDS = (
    "dataset", "sample_id", "image_id", "image_path", "group_id", "split",
    "task_type", "question", "options_json", "reference_answer", "category",
    "image_sha256",
)
GENERATION_FIELDS = (
    "run_id", "dataset", "sample_id", "model_key", "model_id", "model_revision",
    "manifest_sha256", "config_sha256", "raw_output", "parsed_answer",
    "parse_status", "generated_tokens", "stop_reason", "latency_ms",
)
# Reference answers must not be passed to the model-facing input builder.
PARSE_STATUSES = ("parsed", "unparsed")

## 3. Configuration sanity check

TODO: later add GPU/CUDA, available memory, input coverage, exact dependency versions, and checkpoint-access checks.

In [ ]:
# This is a configuration sanity check, not a GPU or data validation.
import platform

if MODEL_KEY not in MODEL_SPECS:
    raise ValueError("Select a model key from MODEL_SPECS.")
if RUN_MODE not in {"prepare", "pilot", "evaluate"}:
    raise ValueError("RUN_MODE must be prepare, pilot, or evaluate.")
if BATCH_SIZE != 1 or DO_SAMPLE is not False:
    raise ValueError("The initial protocol uses batch size 1 and greedy generation.")
if PRECISION not in {None, "fp16", "fp32", "int4_fp32"}:
    raise ValueError("Unknown precision configuration; document new configurations first.")
if RUN_MODE == "evaluate":
    missing = [name for name in (
        "RUN_ID", "DATA_ROOT", "MANIFEST_PATH", "MODEL_REVISION",
        "PROCESSOR_REVISION", "PRECISION",
    ) if globals()[name] is None]
    if not IMPLEMENTATION_VALIDATED:
        missing.append("validated implementation and final protocol freeze")
    if missing:
        raise RuntimeError("Final evaluation is blocked: " + ", ".join(missing))

print({"dataset": DATASET, "model": MODEL_KEY, "mode": RUN_MODE,
       "python": platform.python_version(), "status": "scaffold only"})

## 4. Data audit and frozen manifests — TODO

- Verify archive integrity, source checksum, license, image readability, and metadata joins.
- Verify pathology-to-label mapping and patient identity; do not assume `Case` or filename semantics.
- Review exact/near duplicates and uncertain records with documented, prediction-independent rules.
- Sample 400 distinct-patient images (200 per class), if eligible counts permit.
- Reserve the additional 20-image Gemma pilot (10 per class) with no patient/image/duplicate overlap.
- Fix the Qwen pilot size; the held-out Gemma set may be reused after a recorded decision.
- Save manifests with the common schema and source hashes. Final runs must load them unchanged.

The audit and sampler are intentionally not implemented in the initial scaffold.

## 5. Prompts and strict answer parsing — TODO

Use identical task wording across the four models and their native chat templates.
Decode generated tokens only. Implement the parser locally in this notebook and
keep its rules aligned with the other inference notebook. Accept only a complete,
unambiguous final answer; do not match labels inside arbitrary reasoning.

Add synthetic checks for plain labels/letters, complete answer tags, reasoning
containing misleading labels, conflicting answers, missing closing tags, empty
outputs, unsupported options, and truncation. Freeze normalization and invalid
handling on pilot evidence; retain raw text and parsing diagnostics.

In [ ]:
BUS_BRA_PROMPT = """You are a medical assistant. Examine this breast ultrasound image.
Classify the lesion as exactly ONE of the following:
- benign: non-cancerous lesion
- malignant: cancerous lesion

Respond with ONLY the label: benign or malignant."""

## 6. Model and processor loading — TODO

Implement local Qwen2-VL and Gemma 3 adapters for the selected model key. Pin model
and processor revisions, use native image/chat processing, and record the actual
precision, quantization details, attention backend, image preprocessing, and device
map. Confirm access terms and secrets are configured without logging credentials.
Load only one model per execution. Numerical/output stability must be measured.

## 7. Feasibility and output pilots — TODO

**Qwen pair:** initially test non-quantized FP16 on held-out BUS-BRA samples.
**Gemma pair:** test the same 20 images under FP32 and 4-bit weights with FP32
compute, where feasible. Run one selected checkpoint/configuration per execution;
up to 80 responses cover both Gemma checkpoints and both precision configurations.

Record load failures/OOMs, actual dtypes/device map, peak memory per GPU, runtime,
non-finite logits where measurable, raw responses, token counts, stop reason,
answer validity, and 32-token truncation. Compare each Gemma model's outputs
across precision settings using matching sample IDs. Do not select settings by
pilot accuracy or infer quantization equivalence from 20-image agreement.

## 8. Inference, saving, and resumption — TODO

Use only the selected frozen manifest. Build a separate model-facing input record
that excludes reference answers and privileged metadata. Save each completed
response incrementally as JSONL using `GENERATION_FIELDS`, and save exact run
metadata separately. Keep execution failures/retries separate from completed rows.

Resume only matching dataset/sample, manifest, model revision, and complete config
identities. Detect partial/duplicate records; do not silently skip missing cases.
Persist Kaggle outputs and reattach them for a later session when needed.

The next cell is a deliberate execution stop while implementation is incomplete.

In [ ]:
raise NotImplementedError(
    "Initial scaffold: implement and validate audit, model loading, parsing, "
    "and resumable inference before running experiments."
)

## 9. Validate and export — TODO

Verify exact manifest coverage, unique completed records, matching revisions and
configuration hashes, and no pilot/test contamination. Export raw generation
JSONL, metadata, failures/diagnostics, and a coverage report under `OUTPUT_ROOT`.
No final metrics should be reported for incomplete runs. Scoring and paired
comparisons are performed from saved outputs in `Analyze_Results.ipynb`.

Clear notebook outputs before committing. Keep private dataset records and
credentials out of Git; use permitted storage for experiment exports.